# Proyecto practico: red neuronal con TensorFlow/Keras

In [ ]:
# Importamos las librerias principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Utilizaremos el **Breast Cancer Wisconsin Data Set** de Kaggle: https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data

El objetivo es clasificar la columna **diagnosis** (M = Maligno, B = Benigno).

In [ ]:
# Cargamos el dataset
df = pd.read_csv('data.csv')
df.head()

## Analisis exploratorio de datos

In [ ]:
# Visualizacion del dataframe
df.head(10)

In [ ]:
# Analizamos el shape del objeto
print('Filas:', df.shape[0])
print('Columnas:', df.shape[1])

In [ ]:
# Eliminamos columnas que no aportan informacion
df = df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')
print('Shape luego de limpieza:', df.shape)

In [ ]:
# Verificamos la transformacion
df.head()

In [ ]:
# Visualizamos los tipos de datos
df.dtypes

Primer resumen de los datos:
* Hay 30 variables numericas y 1 variable objetivo categorica (diagnosis).
* diagnosis toma dos valores: M (Maligno) y B (Benigno).
* No existen valores nulos en el dataset.

In [ ]:
# Exploramos la variable objetivo
print(df['diagnosis'].value_counts())
print()
print(df['diagnosis'].value_counts(normalize=True).round(4) * 100)

In [ ]:
# Grafico de distribucion de la variable objetivo
plt.figure(figsize=(6, 4))
df['diagnosis'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'], edgecolor='white')
plt.title('Distribucion de la variable objetivo - diagnosis')
plt.xlabel('Diagnosis')
plt.ylabel('Cantidad')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Verificamos valores nulos
print('Valores nulos por columna:')
print(df.isnull().sum())
print()
print('Total de nulos:', df.isnull().sum().sum())

In [ ]:
# Estadisticas descriptivas de las variables numericas
df.describe()

## Procesamiento de datos

In [ ]:
# Separamos en X e y
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

print('Shape X:', X.shape)
print('Shape y:', y.shape)

In [ ]:
# Codificamos la variable objetivo: M = 1, B = 0
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(y)

print('Clases:', le.classes_)
print('Primeros 10 valores codificados:', y_enc[:10])

In [ ]:
# Importamos librerias para el preprocesamiento y division del dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 20% para test y 80% para train
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.20, random_state=42, stratify=y_enc
)

print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)

In [ ]:
# Veamos que obtuvimos
print('y_train distribucion:')
print(pd.Series(y_train).value_counts())
print()
print('y_test distribucion:')
print(pd.Series(y_test).value_counts())

In [ ]:
# Escalamos los datos (importante para redes neuronales)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('X_train_sc (primeras 3 filas):')
print(X_train_sc[:3])

## Entrenamiento del modelo de clasificacion con red neuronal

In [ ]:
# Importamos TensorFlow y Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)

In [ ]:
# Creacion de la red neuronal
# Arquitectura: 3 capas densas + capa de salida con sigmoid
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_sc.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1,  activation='sigmoid')
], name='RedNeuronal_BreastCancer')

model.summary()

In [ ]:
# Compilamos el modelo
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Entrenamiento
history = model.fit(
    X_train_sc, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    verbose=1
)

In [ ]:
# Calculo de las predicciones en Train y Test
y_pred_prob_train = model.predict(X_train_sc).flatten()
y_pred_prob_test  = model.predict(X_test_sc).flatten()

# Convertimos probabilidades a clases (umbral 0.5)
y_pred_train = (y_pred_prob_train >= 0.5).astype(int)
y_pred_test  = (y_pred_prob_test  >= 0.5).astype(int)

## Evaluacion del modelo de clasificacion con red neuronal

In [ ]:
# Calculo de metricas
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Accuracy en Train
acc_train = accuracy_score(y_train, y_pred_train)
print('Accuracy Train:', round(acc_train, 4), '(' + str(round(acc_train*100, 2)) + '%)')

# Accuracy en Test
acc_test = accuracy_score(y_test, y_pred_test)
print('Accuracy Test: ', round(acc_test, 4), '(' + str(round(acc_test*100, 2)) + '%)')

In [ ]:
# Reporte de clasificacion completo
print('Reporte de clasificacion (Test):')
print(classification_report(y_test, y_pred_test, target_names=['Benigno (B)', 'Maligno (M)']))

In [ ]:
# Curva de perdida (Loss)
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'],     label='Train Loss',      color='tomato')
plt.plot(history.history['val_loss'], label='Val Loss',        color='orange', linestyle='--')
plt.title('Curva de perdida - Loss')
plt.xlabel('Epocas')
plt.ylabel('Binary Crossentropy')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Curva de accuracy
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'],     label='Train Accuracy', color='steelblue')
plt.plot(history.history['val_accuracy'], label='Val Accuracy',   color='orange', linestyle='--')
plt.title('Curva de accuracy')
plt.xlabel('Epocas')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de confusion
cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benigno', 'Maligno'],
            yticklabels=['Benigno', 'Maligno'],
            linewidths=2, linecolor='white',
            annot_kws={'size': 14, 'weight': 'bold'})
plt.title('Matriz de Confusion - Red Neuronal')
plt.ylabel('Valor Real')
plt.xlabel('Prediccion')
plt.tight_layout()
plt.show()

In [ ]:
# Curva ROC
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test, y_pred_prob_test)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='purple', lw=2, label='ROC curve (AUC = ' + str(round(roc_auc, 4)) + ')')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Resumen final de metricas
from sklearn.metrics import precision_score, recall_score, f1_score

print('=' * 45)
print('  RESUMEN FINAL - METRICAS EN TEST')
print('=' * 45)
print('Accuracy  :', round(accuracy_score(y_test, y_pred_test), 4))
print('Precision :', round(precision_score(y_test, y_pred_test), 4))
print('Recall    :', round(recall_score(y_test, y_pred_test), 4))
print('F1-Score  :', round(f1_score(y_test, y_pred_test), 4))
print('ROC-AUC   :', round(roc_auc, 4))
print('=' * 45)